<a href="https://colab.research.google.com/github/speediedan/finetuning-scheduler/blob/main/src/fts_examples/notebooks/publish/gsm8k/fts_gsm8k_pytorch.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>

In [ ]:
# To run this notebook in a fresh environment (Colab, for example), uncomment the line below. The PyTorch API
# needs Fine-Tuning Scheduler 2.15, installed from main until 2.15 is released.
# %pip install 'finetuning-scheduler[examples] @ git+https://github.com/speediedan/finetuning-scheduler.git@main'


# Scheduled Fine-Tuning with the Fine-Tuning Scheduler PyTorch API

![Fine-Tuning Scheduler logo](logo_fts.png){height="55px" width="401px"}

[Fine-Tuning Scheduler](https://finetuning-scheduler.readthedocs.io/en/stable/index.html) (FTS) accelerates and enhances
foundation model experimentation with flexible, multi-phase fine-tuning schedules. A schedule says which parameters to
thaw in each phase; FTS thaws them, adds them to the optimizer's parameter groups (optionally reinitializing the
optimizer or lr scheduler) and decides when each phase ends. Training with FTS is simple and brings several benefits:

- it conveniently abstracts and composes a broad space of fine-tuning methods
- expedites and facilitates exploration of model tuning dynamics
- it composes with interpretability research packages, expanding the space of experimental interventions

FTS runs a schedule through either of two interfaces, which share one scheduling core and one schedule format: the
[FinetuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.fts.html#finetuning_scheduler.fts.FinetuningScheduler) callback for Lightning, and the PyTorch API's [TuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.tuning_scheduler.html#finetuning_scheduler.core.tuning_scheduler.TuningScheduler) for a training loop you write yourself. This notebook uses
the PyTorch API, which is experimental in Fine-Tuning Scheduler 2.15, and does not import Lightning. The
[Lightning version of this tutorial](https://finetuning-scheduler.readthedocs.io/en/latest/examples/notebooks/gsm8k/fts_gsm8k_lightning.html) runs the same experiment with the callback.

Setup is straightforward, just install from PyPI! Since this notebook-based example requires a few additional packages
(e.g. ``transformers``, ``datasets``), install the ``finetuning-scheduler`` package with the ``[examples]`` extra.
For additional installation options, please see the Fine-Tuning Scheduler
[README](https://github.com/speediedan/finetuning-scheduler/blob/main/README.md).



<div style="display:inline">

Fundamentally, [Fine-Tuning Scheduler](https://finetuning-scheduler.readthedocs.io/en/stable/index.html) enables
scheduled, multi-phase, fine-tuning of foundation models. Gradual unfreezing (i.e. thawing) can help maximize
foundation model knowledge retention while allowing (typically upper layers of) the model to
optimally adapt to new tasks during transfer learning [1, 2, 3]

</div>

A [TuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.tuning_scheduler.html#finetuning_scheduler.core.tuning_scheduler.TuningScheduler) orchestrates the gradual unfreezing of a model via a fine-tuning schedule that is either implicitly generated
(the default) or explicitly provided by the user (more computationally efficient). It owns which parameters are
trainable, the optimizer's parameter groups and when the next phase starts; the loop stays yours. Phase transitions are
driven by transition policies such as [PlateauTransition](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.transitions.html#finetuning_scheduler.core.transitions.PlateauTransition) (the PyTorch API's counterpart of the callback's [FTSEarlyStopping](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.fts_supporters.html#finetuning_scheduler.fts_supporters.FTSEarlyStopping)), by the epoch
limit a phase may set (``max_transition_epoch``), or by a composition of the two. A training session completes when the
final phase of the schedule has its stopping criteria met.

![Fine-Tuning Scheduler explicit loss animation](fts_explicit_loss_anim.gif){height="272px" width="376px"}

## Basic Usage

If no fine-tuning schedule is provided (``None``), the [TuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.tuning_scheduler.html#finetuning_scheduler.core.tuning_scheduler.TuningScheduler) generates a
[default schedule](#The-Default-Fine-Tuning-Schedule) and fine-tunes according to it. Build it for your model, call
``setup()`` once, iterate over ``epochs()``, and report each epoch's validation metrics with ``epoch_end()``. It builds
the optimizer from the first phase's parameters and decides when each phase starts; the loop stays yours:

```python
from functools import partial

import torch
from finetuning_scheduler import PlateauTransition, TuningScheduler

fts = TuningScheduler(
    model,
    None,  # the default schedule, or the path to a schedule YAML file, or a mapping
    optimizer=partial(torch.optim.AdamW, lr=1e-3),
    transition=PlateauTransition(monitor="val_loss", patience=2, min_delta=0.01),
)
fts.setup()
for epoch in fts.epochs(max_epochs=30):
    for x, y in train_loader:
        loss = torch.nn.functional.cross_entropy(model(x), y)
        loss.backward()
        fts.optimizer.step()  # always `fts.optimizer`: a phase can replace the optimizer
        fts.optimizer.zero_grad()
    fts.epoch_end({"val_loss": evaluate(model)})
```

## The Default Fine-Tuning Schedule

Schedule definition is facilitated via [gen_ft_schedule](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.schedule.html#finetuning_scheduler.core.schedule.gen_ft_schedule), which dumps a default fine-tuning schedule (by default using a naive,
2-parameters per level heuristic) which can be adjusted as desired by the user and/or subsequently passed to the
[TuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.tuning_scheduler.html#finetuning_scheduler.core.tuning_scheduler.TuningScheduler). Using the default/implicitly generated schedule will likely be less computationally efficient than a
user-defined fine-tuning schedule but is useful for exploring a model's fine-tuning behavior and can serve as a good
baseline for subsequent explicit schedule refinement. While the [TuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.tuning_scheduler.html#finetuning_scheduler.core.tuning_scheduler.TuningScheduler) supports a single optimizer and (optional) lr
scheduler, per-phase maximum learning rates can be set as demonstrated in the next section.

## Specifying a Fine-Tuning Schedule

To specify a fine-tuning schedule, it's convenient to first generate the default schedule and then alter the
thawed/unfrozen parameter groups associated with each fine-tuning phase as desired. Fine-tuning phases are zero-indexed
and executed in ascending order.

1. First, generate the default schedule to a directory of your choice. It will be named after your model's class with
   the suffix ``_ft_schedule.yaml``.

```python
from finetuning_scheduler.core.schedule import gen_ft_schedule

schedule_path = gen_ft_schedule(model, "schedules/")
```

2. Alter the schedule as desired. Here, on the left, are the first phases of the schedule generated for this
   notebook's model, and on the right the schedule we tune it with below (highlighted: what the generated
   schedule does not have).

<table>
<tr><th>Generated: the first 3 of 155 phases</th><th>Explicit: <code>fts_explicit</code>'s 4 phases</th></tr>
<tr><td style="vertical-align:top"><pre>
0:
  params:
  - model.norm.weight
  - model.layers.27.post_attention_layernorm.weight
1:
  params:
  - model.layers.27.input_layernorm.weight
  - model.layers.27.mlp.down_proj.weight
2:
  params:
  - model.layers.27.mlp.up_proj.weight
  - model.layers.27.mlp.gate_proj.weight
...
</pre></td><td style="vertical-align:top"><pre>
0:
  params:
<mark>  - role:adapter # the LoRA adapters first</mark>
<mark>  max_transition_epoch: 3</mark>
1:
  params:
  - model\.layers\.(2[4-7])\.(?!.*lora_).*
  - model\.norm\.weight
<mark>  lr: 2.0e-05</mark>
2:
  params:
  - model\.layers\.(1[6-9]|2[0-3])\.(?!.*lora_).*
3:
  params:
  - model\.layers\.([0-9]|1[0-5])\.(?!.*lora_).*
</pre></td></tr>
</table>

3. Once the fine-tuning schedule has been altered as desired, pass it to the [TuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.tuning_scheduler.html#finetuning_scheduler.core.tuning_scheduler.TuningScheduler) to commence scheduled training (a
   mapping with the same content works too):

```python
fts = TuningScheduler(model, "/path/to/my/schedule/my_schedule.yaml", optimizer=..., transition=...)
```

A schedule names parameters as the model does: a [TuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.tuning_scheduler.html#finetuning_scheduler.core.tuning_scheduler.TuningScheduler) is given the model itself, so its schedules have no ``model.``
prefix, which the Lightning tutorial's schedules carry because there the model is an attribute of a
``LightningModule``.

## Early-Stopping and Epoch-Driven Phase Transition Criteria

Early-stopping (plateau) and epoch-driven transition criteria are composed, as they are under Lightning: a
[PlateauTransition](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.transitions.html#finetuning_scheduler.core.transitions.PlateauTransition) is the PyTorch API's counterpart of the callback's ``FTSEarlyStopping``. If a ``max_transition_epoch`` is specified for a
given phase, the next fine-tuning phase will begin at that epoch unless the early-stopping criteria (a plateau in
``val_loss``, say) are met first. For transitions that are exclusively epoch-driven (the callback's
``epoch_transitions_only``), pass no plateau policy: every phase but the final one then sets its
``max_transition_epoch``, and the final phase runs to ``max_epochs`` (or to an [EpochTransition](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.transitions.html#finetuning_scheduler.core.transitions.EpochTransition)). Several policies can be
passed as a list or composed with [AnyOf](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.transitions.html#finetuning_scheduler.core.transitions.AnyOf), and a policy can also end a phase at a step count or on any condition you
write (see [Transitions](https://finetuning-scheduler.readthedocs.io/en/latest/pytorch/plain_pytorch_loop.html#transitions)).

<div class="alert alert-info">

**Tip:** Use of regex expressions can be convenient for specifying more complex schedules. Also, a per-phase base
maximum lr can be specified:

<pre>
0:
<mark>  params: # the parameters for each phase definition can be fully specified</mark>
  - model.norm.weight
  - model.layers.27.post_attention_layernorm.weight
  max_transition_epoch: 3
1:
<mark>  params: # or specified via a regex</mark>
  - model\.layers\.2[4-7]\.mlp\..*
2:
  params:
  - model\.layers\.2[4-7]\.self_attn\.(?!.*lora_).*
  max_transition_epoch: 9
<mark>  lr: 1e-06 # per-phase maximum learning rates can be specified</mark>
3:
<mark>  params: # names, regexes and role selectors can be used in the same phase</mark>
  - model.layers.27.input_layernorm.weight
  - model\.layers\.2[4-6]\.input_layernorm\.weight
  - role:adapter
</pre>

</div>

The end-to-end example in this notebook ([Scheduled Fine-Tuning for Math Reasoning](#Scheduled-Fine-Tuning-for-Math-Reasoning))
uses the [TuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.tuning_scheduler.html#finetuning_scheduler.core.tuning_scheduler.TuningScheduler) in explicit mode to fine-tune a small language model on
[GSM8K](https://huggingface.co/datasets/openai/gsm8k) math word problems, measuring what it retains as it goes.

## Resuming Scheduled Fine-Tuning Training Sessions

``save()`` writes the full training state for an exact resume: the model, optimizer and lr scheduler, the
[TuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.tuning_scheduler.html#finetuning_scheduler.core.tuning_scheduler.TuningScheduler)'s own state (its phase, its transition policies' state and its checkpoint bookkeeping), every random stream, and
anything registered with ``register_stateful`` (a sampler or a data position, for example). To resume, build the same
model, schedule and [TuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.tuning_scheduler.html#finetuning_scheduler.core.tuning_scheduler.TuningScheduler) in a fresh process, then call ``setup()`` and ``load()``:

```python
fts.register_stateful("sampler", sampler)  # anything with state_dict() and load_state_dict()
fts.save("some/path/to/my_state.pt")

# later, in a fresh process, after building the same model, schedule and tuning scheduler
fts.setup()
fts.load("some/path/to/my_state.pt")
for epoch in fts.epochs(max_epochs=30):  # continues from the saved phase and epoch
    ...
```

Training will resume at the depth/level of the saved state according to the specified schedule. Schedules can be
altered between training sessions but schedule compatibility is left to the user for maximal flexibility. If executing
a user-defined schedule, typically the same schedule should be provided for the original and resumed training sessions.

By default (``restore_best`` is ``True`` whenever a checkpoint policy such as [BestCheckpoint](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.checkpoint.html#finetuning_scheduler.core.checkpoint.BestCheckpoint) is given), the [TuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.tuning_scheduler.html#finetuning_scheduler.core.tuning_scheduler.TuningScheduler) restores the
best available checkpoint before each fine-tuning depth transition, so every phase starts from the best state of the
one before.

A save keeps the checkpoints it refers to until the next save, so resuming from the latest save always finds them. An
older save can be resumed while its checkpoints remain; if its best checkpoint has been removed, ``load()`` raises
``FileNotFoundError`` rather than restoring the wrong state later.

<div class="alert alert-warning">

**Note:** The [TuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.tuning_scheduler.html#finetuning_scheduler.core.tuning_scheduler.TuningScheduler) schedules models you shard yourself, before handing them to it: FSDP2 (``fully_shard``), tensor
parallelism (``parallelize_module``) or both, calling it on every rank (see
[Distributed Training](https://finetuning-scheduler.readthedocs.io/en/latest/pytorch/plain_pytorch_loop.html#distributed-training) and the [model-parallel example](https://finetuning-scheduler.readthedocs.io/en/latest/pytorch/model_parallel_plain_loop.html)).
``DistributedDataParallel`` (and the composable ``replicate()``) and ``ZeroRedundancyOptimizer`` are refused: DDP
registers only the parameters that require grad when it is built, so parameters thawed in later phases would never be
gradient-synchronized. With FSDP2, they stay synchronized.
</div>

## Scheduled Fine-Tuning for Math Reasoning

The following example demonstrates the use of the [TuningScheduler](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.tuning_scheduler.html#finetuning_scheduler.core.tuning_scheduler.TuningScheduler) to fine-tune a small language model,
[Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B), on [GSM8K](https://huggingface.co/datasets/openai/gsm8k), a set
of grade-school math word problems with worked solutions. It runs the same experiment as the
[Lightning tutorial](https://finetuning-scheduler.readthedocs.io/en/latest/examples/notebooks/gsm8k/fts_gsm8k_lightning.html), in a training loop of our own.

Fine-tuning teaches a model a task, and it can also make the model forget some of what it knew before. Gradual
unfreezing is one answer to that [1, 2, 3]: train the parameters closest to the output first, and reach the rest of
the model only when the task needs them. To see the trade, every run here measures two losses at the end of each epoch:

- ``val_loss``, the task loss: the model's loss on the worked solutions of held-out GSM8K problems;
- ``retention_loss``, the model's loss on ordinary encyclopedic text
  ([WikiText-103](https://huggingface.co/datasets/Salesforce/wikitext)) it was not tuned on. It starts at the
  pretrained model's value, and a rise means the model is losing some of its general language modeling.

The schedule also thaws a set of low-rank adapters ([LoRA](https://arxiv.org/abs/2106.09685)), which we add to the
model ourselves in a dozen lines, so the example also shows how a schedule addresses parameters by their *role*
rather than by their names. Every run logs to TensorBoard under the same metric names the Lightning tutorial's
``TensorBoardLogger`` uses, so the two tutorials' training curves can be compared directly.

In [ ]:
# papermill parameters: shrink these for a quick smoke run
MODEL_NAME = "Qwen/Qwen3-0.6B"
MAX_EPOCHS = 16
TRAIN_SUBSET = 1000  # GSM8K training problems to tune on (of 7,473; the docs render uses 4000); None keeps them all
VAL_SUBSET = 128  # held-out GSM8K problems the task loss is measured on
RETENTION_WINDOWS = 64  # 512-token WikiText-103 windows the retention loss is measured on
BATCH_SIZE = 8
SEED = 42
AMP_DTYPE = "auto"  # "auto" uses bfloat16 where the GPU supports it, else float16 (a Colab T4, for example)

In [ ]:
import warnings

# datasets and transformers import tqdm, which warns once that ipywidgets is missing (its text progress bars
# work without it), so filter that before the imports below
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
import gc
import math
import os
import warnings
from functools import partial

import datasets
import torch
import yaml
from tensorboardX import SummaryWriter
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import logging as transformers_logging

from finetuning_scheduler import CanonicalView, PlateauTransition, TuningScheduler
from finetuning_scheduler.core.checkpoint import BestCheckpoint
from finetuning_scheduler.core.schedule import gen_ft_schedule

transformers_logging.set_verbosity_error()
datasets.logging.set_verbosity_error()
datasets.logging.disable_progress_bar()
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
if AMP_DTYPE == "auto":
    bf16 = use_amp and torch.cuda.is_bf16_supported(including_emulation=False)
    amp_dtype = torch.bfloat16 if bf16 else torch.float16
else:
    amp_dtype = {"bf16": torch.bfloat16, "fp16": torch.float16}[AMP_DTYPE]
print(f"Running on {device}" + (f" with {amp_dtype} autocast" if use_amp else " (this notebook needs a GPU)"))

### Data

Each GSM8K example becomes a prompt (``Question: ...`` followed by ``Answer:``) and the worked solution. The loss is
taken on the solution's tokens only: the prompt's labels are ``-100``, which the model's loss ignores. The retention
windows are 512-token stretches of WikiText-103's validation text, scored with the usual next-token loss.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 384  # tokens: longer examples (a few dozen of GSM8K's) are truncated


def encode(example) -> dict[str, list[int]]:
    prompt = tokenizer(f"Question: {example['question']}\nAnswer: ")["input_ids"]
    answer = tokenizer(example["answer"] + tokenizer.eos_token)["input_ids"]
    return {"input_ids": (prompt + answer)[:MAX_LEN], "labels": ([-100] * len(prompt) + answer)[:MAX_LEN]}


def collate(rows) -> dict[str, torch.Tensor]:
    """Right-pad a batch; padding is masked out of attention and of the loss."""
    n = max(len(r["input_ids"]) for r in rows)
    ids = torch.full((len(rows), n), tokenizer.pad_token_id)
    labels = torch.full((len(rows), n), -100)
    mask = torch.zeros((len(rows), n), dtype=torch.long)
    for i, r in enumerate(rows):
        k = len(r["input_ids"])
        ids[i, :k], labels[i, :k], mask[i, :k] = torch.tensor(r["input_ids"]), torch.tensor(r["labels"]), 1
    return {"input_ids": ids, "labels": labels, "attention_mask": mask}


gsm8k = datasets.load_dataset("openai/gsm8k", "main")
train_split = gsm8k["train"].shuffle(seed=SEED)
train_rows = [encode(e) for e in train_split.select(range(TRAIN_SUBSET or len(train_split)))]
val_rows = [encode(e) for e in gsm8k["test"].select(range(VAL_SUBSET))]
wiki = datasets.load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="validation")
wiki_ids = tokenizer("\n".join(t for t in wiki["text"][:4000] if t.strip()))["input_ids"]
retention_windows = torch.tensor([wiki_ids[i * 512:(i + 1) * 512] for i in range(RETENTION_WINDOWS)])


def train_loader() -> DataLoader:
    # a seeded shuffle, made afresh for each run: every scenario sees the same order, whatever ran before it
    return DataLoader(train_rows, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate,
                      generator=torch.Generator().manual_seed(SEED))


print(f"{len(train_rows)} training and {len(val_rows)} validation problems, {len(retention_windows)} retention windows")

### Adding LoRA Adapters

A LoRA adapter leaves a linear layer's weight $W$ frozen and learns a low-rank update beside it: the layer computes
$W x + \frac{\alpha}{r} B A x$, with $B$ of shape $d_{out} \times r$ and $A$ of shape $r \times d_{in}$ for a small rank
$r$. $B$ starts at zero, so the adapted model starts out identical to the pretrained one. We wrap the query and value
projections of every attention block, which adds about 2.3 million parameters to the model's 596 million.

The adapters' parameters are named like any others (``model.layers.0.self_attn.q_proj.lora_a``), so a schedule could
list them by regex. We give them a *role* instead, and a schedule phase selects them with ``role:adapter``. This is how
that entry becomes a set of parameters:

1. ``add_lora`` returns a mapping from each adapter tensor's name to its role, ``adapter``: 112 tensors, the
   ``lora_a`` and ``lora_b`` of the query and value projections in each of the 28 decoder blocks.
2. A [CanonicalView](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.views.html#finetuning_scheduler.core.views.CanonicalView) over the model carries that mapping (``CanonicalView(model, roles=roles)``). The view keeps the
   model's own parameter names, and every parameter it is not given a role for has the role ``base``.
3. When the tuning scheduler starts (``setup()``), it compiles the schedule through the view. A ``params`` entry that
   starts with ``role:`` is replaced by every parameter whose role matches it: ``role:adapter`` matches the role
   ``adapter`` and any named one (``adapter:<name>``), while ``role:adapter:<name>`` would match only the adapters
   given that name. Every other entry is a parameter name or a regular expression matched against the names. So
   ``role:adapter`` compiles to exactly the 112 adapter tensors, and a model with more or fewer adapters, or adapters
   elsewhere, needs no change to the schedule.
4. A parameter can be thawed by only one phase: the scheduler refuses a schedule in which two phases claim the same
   parameter. That is why the block entries in the schedule below exclude the adapter tensors inside those blocks.

The compiled schedule can be inspected after ``setup()``: ``fts.compiled_schedule.physical_for(depth)`` lists the
parameters a phase thaws.

In [ ]:
LORA_RANK = 16


class LoRALinear(nn.Module):
    """A frozen linear layer plus a trainable low-rank update (``lora_a`` and ``lora_b``)."""

    def __init__(self, base: nn.Linear, rank: int = LORA_RANK, alpha: int = 2 * LORA_RANK):
        super().__init__()
        self.base, self.scale = base, alpha / rank
        self.lora_a = nn.Parameter(torch.randn(rank, base.in_features) / math.sqrt(rank))
        self.lora_b = nn.Parameter(torch.zeros(base.out_features, rank))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.base(x) + (x @ self.lora_a.t() @ self.lora_b.t()) * self.scale


def add_lora(model: nn.Module) -> dict[str, str]:
    """Wrap every attention block's query and value projections; return the adapter tensors' roles."""
    for layer in model.model.layers:
        for name in ("q_proj", "v_proj"):
            setattr(layer.self_attn, name, LoRALinear(getattr(layer.self_attn, name)))
    return {n: "adapter" for n, _ in model.named_parameters() if ".lora_" in n}

### Our Training Sessions

We'll be comparing two fine-tuning training configurations. Both depend upon a shared set of defaults, only differing
in their respective fine-tuning schedules.

| Experiment Tag    | Training Scenario Description                                                        |
|:-----------------:| ------------------------------------------------------------------------------------ |
| ``fts_explicit``  | Training with a fine-tuning schedule explicitly provided by the user                   |
| ``nofts_baseline``| A baseline fine-tuning training session (without scheduled fine-tuning or adapters)    |

Let's begin by generating the default schedule for our model, as step 1 of
[Specifying a Fine-Tuning Schedule](#Specifying-a-Fine-Tuning-Schedule) does, and looking at its first phases.

In [ ]:
schedule_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
default_schedule_path = gen_ft_schedule(schedule_model, os.path.join("fts_logs", "generated"))
del schedule_model
with open(default_schedule_path) as f:
    default_schedule = yaml.safe_load(f)
print(f"{len(default_schedule)} phases in {default_schedule_path}, the first three:\n")
print(yaml.safe_dump({k: default_schedule[k] for k in list(default_schedule)[:3]}, sort_keys=False))

The default schedule thaws two parameters per phase from the last registered one: the final norm and the last
decoder block's normalization weights first, then on down through 28 decoder blocks a pair at a time. That is useful
for exploring a model's tuning dynamics, but its phases are far too fine for a 28-block model, so ``fts_explicit``
groups them into four phases instead, starting with the adapters:

0. the LoRA adapters (``role:adapter``) alone, for at most 3 epochs (``max_transition_epoch``), at the optimizer's
   learning rate;
1. the top 4 decoder blocks (24 to 27) and the final norm, with a phase learning rate of its own (``lr``);
2. the next 8 blocks down (16 to 23);
3. the remaining blocks (0 to 15).

Each phase adds to the ones before it, so the last phase trains the adapters, every decoder block and the final norm.
The ``nofts_baseline`` scenario trains every decoder block and the final norm from the start, in one phase; its model
has no adapters.

The token embeddings stay frozen throughout, in the baseline too (Qwen3-0.6B ties its output layer to them, and
their 156 million parameters' optimizer state would not fit beside the rest on a 16 GB GPU, such as the Colab
T4 these tutorials are sized to run on). Entries are regular
expressions; ``(?!.*lora_)`` keeps the block entries from also matching the adapters inside those blocks, so the
adapters are thawed by their own phase. Phase 0 trains at the optimizer's learning rate (a schedule cannot set
phase 0's ``lr``); phases after the first take their learning rate from the phase's ``lr`` or, without one, from
the tuning scheduler's ``base_max_lr``. As in the Lightning tutorial, we write the schedule to a
file to simulate loading an explicitly defined fine-tuning schedule.

In [ ]:
# Please see the [Fine-Tuning Scheduler documentation](https://finetuning-scheduler.readthedocs.io/en/stable/index.html)
# for a full description of the schedule format
ft_schedule_yaml = r"""
0:
  params:
  - role:adapter
  max_transition_epoch: 3
1:
  params:
  - model\.layers\.(2[4-7])\.(?!.*lora_).*
  - model\.norm\.weight
  lr: 2.0e-05
2:
  params:
  - model\.layers\.(1[6-9]|2[0-3])\.(?!.*lora_).*
3:
  params:
  - model\.layers\.([0-9]|1[0-5])\.(?!.*lora_).*
"""
os.makedirs("fts_logs", exist_ok=True)
explicit_schedule = os.path.join("fts_logs", "Qwen3ForCausalLM_ft_schedule_explicit.yaml")
with open(explicit_schedule, "w") as f:
    f.write(ft_schedule_yaml)
# one phase: every decoder block and the final norm trainable from the start, the embeddings frozen as above
baseline_schedule = {0: {"params": [r"model\.layers\..*", r"model\.norm\.weight"]}}

### Optimizer Configuration

Though other optimizers can arguably yield some marginal advantage contingent on the context,
the Adam optimizer (and the [AdamW version](https://pytorch.org/docs/stable/generated/torch.optim.AdamW.html) which
implements decoupled weight decay) remains robust to hyperparameter choices and is commonly used for fine-tuning
foundation language models. See (Sivaprasad et al., 2020) and (Mosbach, Andriushchenko & Klakow, 2020) for theoretical
and systematic empirical justifications of Adam and its use in fine-tuning large transformer-based language models.

The optimizer is passed to the tuning scheduler as a factory: it builds the optimizer from phase 0's parameter groups
and adds a group as each later phase thaws parameters, so the factory's learning rate is phase 0's. That is
``ADAPTER_LR`` for ``fts_explicit``, whose phase 0 is the adapters, and ``LR`` for the baseline. The top blocks' phase
sets ``LR`` as its own ``lr``, and ``base_max_lr`` sets the lower block phases' to half of it, so lower blocks move
more gently. The usual LoRA learning rate is about ten times the full fine-tuning one, but it made these adapters
forget the most in a short trial, which is why they use a lower one. The learning rate stays constant within a
phase. An lr scheduler can be passed as a factory too (``lr_scheduler``), and reinitialized at phase transitions
(see [LR scheduler reinitialization](https://finetuning-scheduler.readthedocs.io/en/latest/advanced/lr_scheduler_reinitialization.html)).

In [ ]:
LR, ADAPTER_LR, LATER_PHASE_LR = 2e-5, 5e-5, 1e-5
PHASE0_LR = {"fts_explicit": ADAPTER_LR, "nofts_baseline": LR}  # the optimizer's learning rate is phase 0's

### Transitions and Checkpoints

[PlateauTransition](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.transitions.html#finetuning_scheduler.core.transitions.PlateauTransition) plays the part ``FTSEarlyStopping`` plays under Lightning, with the Lightning tutorial's configuration: a
phase ends when ``val_loss`` has not improved by ``min_delta`` for ``patience`` epochs. [BestCheckpoint](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.checkpoint.html#finetuning_scheduler.core.checkpoint.BestCheckpoint) keeps the best
checkpoint, and the tuning scheduler restores it at each phase boundary, so every phase starts from the best state of
the one before. Only the task loss drives the transitions: the retention loss is measured, never optimized.

In [ ]:
def make_tuning_scheduler(model, schedule, scenario: str, roles: dict[str, str]) -> TuningScheduler:
    return TuningScheduler(
        model,
        schedule,
        optimizer=partial(AdamW, lr=PHASE0_LR[scenario], weight_decay=0.0),
        base_max_lr=LATER_PHASE_LR,
        transition=PlateauTransition(monitor="val_loss", patience=2, min_delta=0.001),
        checkpoint=BestCheckpoint(os.path.join("checkpoints", scenario), monitor="val_loss"),
        view=CanonicalView(model, roles=roles),
        log_dir=os.path.join("fts_logs", scenario),
    )

### The Training Loop

This is an ordinary mixed-precision loop. The tuning scheduler appears in four places: ``setup()`` before training,
``epochs()`` to iterate (it starts the next phase when one is due), ``fts.optimizer`` in place of an optimizer held by
the loop (a schedule may replace the optimizer at a phase transition), and ``epoch_end()`` with the epoch's
validation metrics.

The model keeps float32 weights and runs its forward pass under autocast, in bfloat16 where the GPU supports it and
in float16 otherwise, with a gradient scaler (``GradScaler``) to keep float16 gradients from underflowing. Gradient
checkpointing recomputes each block's activations in the backward pass rather than storing them, which keeps the
baseline (every decoder block trainable) within a 16 GB GPU such as a Colab T4.

Each run seeds itself where its model is built and draws a freshly seeded shuffle of the training data, so no scenario
depends on the ones run before it. The Lightning tutorial is configured the same way, so the two tutorials' runs can
be compared directly.

Each run also writes TensorBoard logs to ``tensorboard_logs/<scenario>``, with the Lightning tutorial's metric
names: ``train_loss`` at every optimizer step, and ``val_loss``, ``retention_loss`` and
``finetuning_schedule_depth`` at the end of each epoch, plus each parameter group's learning rate and the number of
trainable parameters, all against the optimizer step.

In [ ]:
def batches(rows, size: int):
    return (collate(rows[i:i + size]) for i in range(0, len(rows), size))


@torch.no_grad()
def evaluate(model) -> dict[str, float]:
    model.eval()
    with torch.autocast(device.type, dtype=amp_dtype, enabled=use_amp):
        task = [model(**{k: v.to(device) for k, v in b.items()}).loss.item() for b in batches(val_rows, BATCH_SIZE)]
        retention = [model(input_ids=w.to(device), labels=w.to(device)).loss.item()
                     for w in retention_windows.split(BATCH_SIZE)]
    model.train()
    return {"val_loss": sum(task) / len(task), "retention_loss": sum(retention) / len(retention)}


def build_model(scenario: str):
    torch.manual_seed(SEED)  # seed each scenario where its model is built, so none depends on the ones before it
    # fp32 master weights: transformers would otherwise keep the checkpoint's stored dtype (bfloat16)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
    roles = add_lora(model) if scenario == "fts_explicit" else {}
    model.gradient_checkpointing_enable()
    model.config.use_cache = False
    return model.to(device), roles


def run(scenario: str, schedule) -> list[dict]:
    model, roles = build_model(scenario)
    fts = make_tuning_scheduler(model, schedule, scenario, roles)
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp and amp_dtype == torch.float16)
    writer = SummaryWriter(os.path.join("tensorboard_logs", scenario))
    history, global_step, loader = [], 0, train_loader()
    fts.setup()
    if roles:  # the parameters ``role:adapter`` compiled to (``physical_for`` lists any phase's)
        thawed = fts.compiled_schedule.physical_for(0)
        print(f"{scenario:>14} | phase 0 (role:adapter) thaws {len(thawed)} tensors, {thawed[0]} first")
    start = evaluate(model)
    print(f"{scenario:>14} | before tuning | val_loss {start['val_loss']:.4f} | "
          f"retention_loss {start['retention_loss']:.4f}")
    for name, value in start.items():
        writer.add_scalar(name, value, 0)
    for epoch in fts.epochs(max_epochs=MAX_EPOCHS):
        train_loss, steps = 0.0, 0
        for batch in loader:
            with torch.autocast(device.type, dtype=amp_dtype, enabled=use_amp):
                loss = model(**{k: v.to(device) for k, v in batch.items()}).loss
            scaler.scale(loss).backward()
            scaler.step(fts.optimizer)
            scaler.update()
            fts.optimizer.zero_grad(set_to_none=True)
            train_loss, steps, global_step = train_loss + loss.item(), steps + 1, global_step + 1
            writer.add_scalar("train_loss", loss.item(), global_step)
        metrics = evaluate(model)
        fts.epoch_end({"val_loss": metrics["val_loss"]})
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        history.append({"epoch": epoch, "depth": fts.curr_depth, "trainable": trainable, **metrics})
        for name, value in (*metrics.items(), ("finetuning_schedule_depth", fts.curr_depth),
                            ("trainable_params", trainable)):
            writer.add_scalar(name, value, global_step)
        for i, group in enumerate(fts.optimizer.param_groups):
            writer.add_scalar(f"lr-AdamW/pg{i + 1}", group["lr"], global_step)
        print(
            f"{scenario:>14} | epoch {epoch:>2} | phase {fts.curr_depth} | trainable {trainable:>11,} | "
            f"train_loss {train_loss / steps:.4f} | val_loss {metrics['val_loss']:.4f} | "
            f"retention_loss {metrics['retention_loss']:.4f}"
        )
    writer.close()
    print(f"{scenario}: finished at phase {fts.curr_depth} after {len(history)} epochs ({fts.stop_reason or 'done'})")
    del model, fts
    gc.collect()  # the model and the tuning scheduler's state reference each other
    if device.type == "cuda":
        print(f"{scenario}: peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB")
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    return history

### Running the Scenarios

``fts_explicit`` uses the schedule above, and its model carries the adapters. The baseline's single phase makes it
ordinary fine-tuning with early stopping, of every decoder block and the final norm at once.

The copy of this notebook rendered in the documentation tuned on 4,000 problems (``TRAIN_SUBSET=4000``) in
bfloat16 on one RTX 4090, and took 39 minutes in all. The defaults above (1,000 problems) are sized for a Colab T4
in float16: we estimate roughly an hour to an hour and a half there, from the 4090's float16 time. Each run peaked
at 12.2 GiB, most of it the retention loss's logits over Qwen3's 151,936-token vocabulary, within a T4's 15 GiB.

In [ ]:
# optionally start TensorBoard and watch the runs' curves while they train by uncommenting the next 2 lines
# %load_ext tensorboard
# %tensorboard --logdir tensorboard_logs

In [ ]:
results = {
    "fts_explicit": run("fts_explicit", explicit_schedule),
    "nofts_baseline": run("nofts_baseline", baseline_schedule),
}

### Reviewing the Training Trajectory

In the rendered run, each phase of ``fts_explicit`` ended when its validation loss stopped improving, and the next
phase started from that phase's best checkpoint:

| Phase | Thawed | Epochs | Best ``val_loss`` | ``retention_loss`` there |
| --- | --- | --- | --- | --- |
| pretrained | | | 1.428 | 3.129 |
| 0 | blocks 24 to 27, final norm | 0 to 3 | 0.549 | 3.128 |
| 1 | + LoRA adapters (``role:adapter``, ``lr`` 5e-5) | 4 to 6 | **0.506** | 3.280 |
| 2 | + blocks 16 to 23 | 7 to 8 | 0.556 | 3.533 |
| 3 | + blocks 0 to 15 | 9 to 10 | 0.538 | 3.526 |
| ``nofts_baseline`` | every block at once | 0 to 2 | 0.477 | 3.172 |

![fts_explicit's task and retention loss by phase](pytorch_fts_explicit_trajectory.png){width="800px"}

- **Phase 0 does most of the work and forgets nothing.** Tuning only the top four blocks takes the task loss from 1.43
  to 0.55 while the retention loss stays at its pretrained value.
- **The adapters buy the explicit schedule's best task loss** (0.506), at a modest retention cost. Thawing the lower
  blocks after them improves neither loss: each later phase overfits within an epoch or two, and its best epoch is
  worse than phase 1's on both.
- **The sawtooth is the best-checkpoint restore.** Each phase starts from the previous phase's best epoch, not its last,
  so the curves drop back at every boundary.

![Task loss against retention loss, each phase's best epoch](pytorch_retention_tradeoff.png){width="700px"}

Plotting each phase's best epoch in (task loss, retention loss) shows the trade the schedule makes. In this run,
ordinary fine-tuning with early stopping did well too: its best epoch, the first, reached a lower task loss than any
phase of the schedule (0.477), with less forgetting than phases 1 to 3. Like every phase's, its forgetting climbs
quickly once it overfits (a retention loss of 3.59 by its third epoch). What the schedule buys here is control rather than a better end
point: phase 0 gives a checkpoint that has learned most of the task without forgetting, and each later phase is a
deliberate, measurable step along the trade-off, which you can stop at.

Expect run-to-run variation: compare scenarios over several seeds (``SEED`` in the parameters cell) before reading much
into a small difference.

To view your own run's curves, run ``tensorboard --logdir tensorboard_logs`` (``tensorboard`` installs separately) or
uncomment the cell above.

## Where to Go Next

- [Scheduled fine-tuning in a plain PyTorch loop](https://finetuning-scheduler.readthedocs.io/en/latest/pytorch/plain_pytorch_loop.html):
  transition policies, phase hooks and gates, exact resumption and distributed training.
- [Model-parallel scheduled fine-tuning in a plain loop](https://finetuning-scheduler.readthedocs.io/en/latest/pytorch/model_parallel_plain_loop.html):
  the same tuning scheduler with FSDP2 and tensor parallelism.
- The [Lightning version of this tutorial](https://finetuning-scheduler.readthedocs.io/en/latest/examples/notebooks/gsm8k/fts_gsm8k_lightning.html), for the
  ``FinetuningScheduler`` callback.

## Footnotes

- [Howard, J., & Ruder, S. (2018)](https://arxiv.org/pdf/1801.06146.pdf). Fine-tuned Language
 Models for Text Classification. ArXiv, abs/1801.06146. [↩](#Scheduled-Fine-Tuning-with-the-Fine-Tuning-Scheduler-PyTorch-API)
- [Chronopoulou, A., Baziotis, C., & Potamianos, A. (2019)](https://arxiv.org/pdf/1902.10547.pdf).
 An embarrassingly simple approach for transfer learning from pretrained language models. arXiv
 preprint arXiv:1902.10547. [↩](#Scheduled-Fine-Tuning-with-the-Fine-Tuning-Scheduler-PyTorch-API)
- [Peters, M. E., Ruder, S., & Smith, N. A. (2019)](https://arxiv.org/pdf/1903.05987.pdf). To tune or not to
 tune? adapting pretrained representations to diverse tasks. arXiv preprint arXiv:1903.05987. [↩](#Scheduled-Fine-Tuning-with-the-Fine-Tuning-Scheduler-PyTorch-API)
- [Sivaprasad, P. T., Mai, F., Vogels, T., Jaggi, M., & Fleuret, F. (2020)](https://arxiv.org/pdf/1910.11758.pdf).
 Optimizer benchmarking needs to account for hyperparameter tuning. In International Conference on Machine Learning
(pp. 9036-9045). PMLR. [↩](#Optimizer-Configuration)
- [Mosbach, M., Andriushchenko, M., & Klakow, D. (2020)](https://arxiv.org/pdf/2006.04884.pdf). On the stability of
fine-tuning bert: Misconceptions, explanations, and strong baselines. arXiv preprint arXiv:2006.04884. [↩](#Optimizer-Configuration)
- [Cobbe, K., Kosaraju, V., Bavarian, M., et al. (2021)](https://arxiv.org/pdf/2110.14168.pdf). Training verifiers to
solve math word problems. arXiv preprint arXiv:2110.14168. [↩](#Scheduled-Fine-Tuning-for-Math-Reasoning)
- [Hu, E. J., Shen, Y., Wallis, P., et al. (2021)](https://arxiv.org/pdf/2106.09685.pdf). LoRA: Low-rank adaptation of
large language models. arXiv preprint arXiv:2106.09685. [↩](#Adding-LoRA-Adapters)